# 10 — Entraînement par imitation (PyTorch)

Vrai entraînement supervisé, **uniquement si** tes sessions contiennent les actions humaines `steer_cmd`, `throttle_cmd`, `brake_cmd`. Si elles sont absentes, le notebook arrête proprement au lieu d’inventer des labels.

In [ ]:
from pathlib import Path
import sys
CWD=Path.cwd().resolve()
REPO=next((p for p in [CWD,*CWD.parents] if (p/'assetto_corsa_gym'/'assetto_corsa_gym'/'AssettoCorsaEnv').is_dir()),None)
if REPO is None:
    raise FileNotFoundError('Ouvre le depot IA_projet_Ouarti_ds_github dans VS Code : son dossier assetto_corsa_gym doit exister.')
PROJECT=REPO/'OUARTI'
if not PROJECT.is_dir():
    raise FileNotFoundError('Dossier OUARTI absent : extraire OUARTI a la racine du depot.')
GYM_ROOT=REPO/'assetto_corsa_gym'/'assetto_corsa_gym'
print('Depot :',REPO)
print('Notebooks :',PROJECT)
print('Python :',sys.version.split()[0],sys.executable)


In [ ]:
import json, numpy as np, pandas as pd
from pathlib import Path
files=sorted((PROJECT/'datasets').glob('session_*/telemetry.jsonl'))
if not files:raise FileNotFoundError('Enregistre une session dans 02_Donnees.')
df=pd.DataFrame([json.loads(line) for line in files[-1].read_text(encoding='utf8').splitlines() if line.strip()])
INPUTS=['speed','yaw','angular_velocity_y','accelX','accelY','steerAngle','accStatus','brakeStatus']
TARGETS=['steer_cmd','throttle_cmd','brake_cmd']
missing=[c for c in INPUTS+TARGETS if c not in df]
if missing:
    raise ValueError('Impossible d entrainer par imitation : champs manquants '+str(missing)+'. Ajouter un enregistrement reel des commandes du conducteur.')
df=df[INPUTS+TARGETS].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan).dropna()
if len(df)<100:raise ValueError('Il faut au moins 100 echantillons etiquetes.')
print(len(df),'exemples de conduite humaine reels.')


In [ ]:
import torch,torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
X=df[INPUTS].to_numpy(np.float32);Y=df[TARGETS].to_numpy(np.float32)
cut=int(len(X)*.8)
mean=X[:cut].mean(axis=0);std=X[:cut].std(axis=0).clip(min=1.e-5)
X=(X-mean)/std
X_train=torch.tensor(X[:cut]); Y_train=torch.tensor(Y[:cut])
X_val=torch.tensor(X[cut:]); Y_val=torch.tensor(Y[cut:])
model=nn.Sequential(nn.Linear(len(INPUTS),64),nn.ReLU(),nn.Linear(64,64),nn.ReLU(),nn.Linear(64,3))
optimizer=torch.optim.Adam(model.parameters(),lr=3e-4)
loss_fn=nn.MSELoss()
for epoch in range(30):
    model.train()
    for xb,yb in DataLoader(TensorDataset(X_train,Y_train),batch_size=64,shuffle=True):
        optimizer.zero_grad();loss=loss_fn(model(xb),yb);loss.backward();optimizer.step()
    model.eval()
    with torch.no_grad():v=float(loss_fn(model(X_val),Y_val))
    if epoch%5==0:print('Epoch',epoch,'validation MSE',round(v,6))
checkpoint=PROJECT/'models'/'imitation_pilote.pt'
torch.save({'state_dict':model.state_dict(),'inputs':INPUTS,'targets':TARGETS,
            'mean':mean,'std':std,'architecture':[64,64]},checkpoint)
print('Modele sauvegarde :',checkpoint)
